# Temel NLP Görevleri

Table of Content:

- [1. Metin Sınıflandırma](#1.-Metin-Sınıflandırma:)
- [2. Varlık İsmi Tanıma](#2.-Varlık-İsmi-Tanıma:)
- [3. Morfolojik Analiz](#3.-Morfolojik-Analiz:)

## 1. Metin Sınıflandırma:


In [3]:
import pandas as pd

spam_df = pd.read_csv("SMSSpamCollection.csv", names=["class", "sms"])

In [13]:
import re
from bs4 import BeautifulSoup as bs
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

stop_words_eng = stopwords.words("english")

def clean_text(text):
    text = text.lower()
    text = bs(text, "html.parser").get_text()

    # Text temizliği:
    # Kelimeler arasındaki '-' karakterleri
    # Lookarounds:   (?=) - positive lookahead
    #                (?!) - negative lookahead
    #                (?<=) - positive lookbehind
    #                (?<!) - negative lookbehind
    text = re.sub(r"(?<=\w)-(?=\w)", " ", text)
    # Harf olmayan karakterlerin tamamı:
    text = re.sub(r"[^A-Za-z\s]", "", text)
    # Peşpeşe birden fazla kez gelen boşluk karakterleri:
    text = re.sub(r"\s{2,}", "", text)
    # Sık kullanılan farklı yazımlar: \b => word boundary 
    text = re.sub(r"cannot", "can not", text)
    text = re.sub(r"\bim\b", "i am", text)

    ttokens = text.split()
    
    text = " ".join([lemmatizer.lemmatize(word) for word in ttokens if word not in stop_words_eng])

    return text

In [17]:
X = spam_df["sms"].apply(clean_text)
y = spam_df["class"]

In [18]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size = 0.33, random_state=60)

In [20]:
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer()
Xtr_BoW = cv.fit_transform(x_train)

In [22]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier()
dt.fit(Xtr_BoW, y_train)

Xte_BoW = cv.transform(x_test)

In [25]:
from sklearn.metrics import confusion_matrix

In [31]:
pred = dt.predict(Xte_BoW)
c_matrix = confusion_matrix(y_test, pred)
tn = c_matrix[0][0]
fn = c_matrix[0][1]
fp = c_matrix[1][0]
tp = c_matrix[1][1]
print(f"{c_matrix}\nPozitive Error Rate: %{(fn/(tn+fn))*100}\nNegative Error Rate: %{(fp/(tp+fp))*100}\nAccuracy: %{100-((fp+fn)/(fp+fn+tn+tp))*100}")

[[1553   22]
 [  63  202]]
Pozitive Error Rate: %1.3968253968253967
Negative Error Rate: %23.77358490566038
Accuracy: %95.3804347826087


## 2. Varlık İsmi Tanıma:

In [41]:
!pip install "click<8.1.0"

  Attempting uninstall: click
    Found existing installation: click 8.4.1
    Uninstalling click-8.4.1:
      Successfully uninstalled click-8.4.1


In [42]:
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     - -------------------------------------- 0.5/12.8 MB 1.5 MB/s eta 0:00:09
     -- ------------------------------------- 0.8/12.8 MB 1.5 MB/s eta 0:00:09
     --- ------------------------------------ 1.0/12.8 MB 1.5 MB/s eta 0:00:08
     ---- ----------------------------------- 1.6/12.8 MB 1.5 MB/s eta 0:00:08
     ----- ---------------------------------- 1.8/12.8 MB 1.5 MB/s eta 0:00:08
     ------ --------------------------------- 2.1/12.8 MB 1.5 MB/s eta 0:00:08
     ------- -------------------------------- 2.4/12.8 MB 1.5 MB/s eta 0:00:07
     -------- ------------------------------- 2.6/12.8 MB 1.5 MB/s eta 0:00:07
     --------- ------------------------------ 3.1/12.8 MB 1.5 MB/s eta 0:00:07
     ---------- ----------------------------- 3.4/12.8 MB 1.5 MB/s eta 0:00:07
     ----------- ---------------------------- 3.7/12.8 MB 1.5 MB/s

In [2]:
import sys
print(sys.executable)

C:\Users\mrtke\anaconda3\python.exe


In [3]:
import sys
!{sys.executable} -m pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     - -------------------------------------- 0.5/12.8 MB 1.5 MB/s eta 0:00:09
     -- ------------------------------------- 0.8/12.8 MB 1.5 MB/s eta 0:00:08
     --- ------------------------------------ 1.0/12.8 MB 1.5 MB/s eta 0:00:08
     ---- ----------------------------------- 1.3/12.8 MB 1.5 MB/s eta 0:00:08
     ----- ---------------------------------- 1.8/12.8 MB 1.5 MB/s eta 0:00:08
     ------ --------------------------------- 2.1/12.8 MB 1.5 MB/s eta 0:00:08
     ------- -------------------------------- 2.4/12.8 MB 1.5 MB/s eta 0:00:07
     -------- ------------------------------- 2.6/12.8 MB 1.5 MB/s eta 0:00:07
     --------- ------------------------------ 2.9/12.8 MB 1.5 MB/s eta 0:00:07
     --------- ------------------------------ 3.1/12.8 MB 1.5 MB/s eta 0:00:07
     ----------- ---------------------------- 3.7/12.8 MB 1.5 MB/s

  error: subprocess-exited-with-error
  
  installing build dependencies for spacy did not run successfully.
  exit code: 1
  
  [570 lines of output]
  Ignoring numpy: markers 'python_version < "3.9"' don't match your environment
    Using cached cymem-2.0.13-cp313-cp313-win_amd64.whl.metadata (9.9 kB)
    Using cached preshed-3.0.13-cp313-cp313-win_amd64.whl.metadata (5.4 kB)
    Using cached murmurhash-1.0.15-cp313-cp313-win_amd64.whl.metadata (2.3 kB)
    Installing build dependencies: started
    Installing build dependencies: finished with status 'error'
    error: subprocess-exited-with-error
  
    installing build dependencies for thinc did not run successfully.
    exit code: 1
  
    [545 lines of output]
    Ignoring numpy: markers 'python_version < "3.9"' don't match your environment
      Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
      Using cached Cython-0.29.37-py2.py3-none-any.whl.metadata (3.1 kB)
      Using cached murmurhash-1.0.15-cp313-cp31

In [1]:
import spacy
nlp = spacy.load("en_core_web_sm")
content = "I work at McDonald's and live in Wyoming. I would like to move to Los Angeles and work at Burger King this year."
doc = nlp(content)

for ent in doc.ents:
    print(ent.text, ent.label_)

entities = [(ent.text, ent.text, ent.label_, ent.lemma_) for ent in doc.ents]

OSError: [E050] Can't find model 'en_core_web_sm'. It doesn't seem to be a Python package or a valid path to a data directory.

## 3. Morfolojik Analiz: